# NPC-coordinate sensitivity experiments

Simulated Gaussian gene profiles, not observed NPC expression or LR p-values.
The current axis-wise partition and its native k-means-label order are the reference.
Change one approximation setting at a time; reuse the same data and initialization within
each replicate. Exact likelihoods at the generating parameters separate approximation
error from changes caused by refitting.

This is a model-matched experiment. It does not replace count-data or biological validation.
Run from the SparseAEH repository, using the same environment as `ll_test.ipynb`
(and `requirements-benchmark.txt`). No TensorFlow or GPU is needed.

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import json
import random
import time
import gc
from pathlib import Path
from scipy.linalg import solve_triangular
from scipy.optimize import linear_sum_assignment
from scipy.special import logsumexp
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics.pairwise import rbf_kernel
from scipy.spatial import distance_matrix
from threadpoolctl import threadpool_limits
from SparseAEH import Kernel, MixedGaussian, GaussianNLL
from SparseAEH.benchmark_logging import BenchmarkRecorder


## Settings

Coordinates are centered and divided by one common spatial extent, preserving distances
up to a single scale factor. LENGTH_SCALES are in these normalized coordinate units:
the RBF kernel is exp(-distance**2 / (2 * length_scale**2)), so SparseAEH's
`l` argument receives `gamma = 1 / (2 * length_scale**2)`, not the length scale itself.

The defaults use a fixed subset of at most 600 NPC locations and 120 simulated genes.
Replicate seeds are SEED + replicate; all methods share the replicate seed.
Partition/order seeds are explicit experimental factors, independent of the data seed.
THREADS controls numerical-library threads, not the server's allocated CPUs.

RUN_DENSE_FIT enables a one-block exact mixture fit using the same implementation and
line search. It is on by default. Each dataset has N_RESTARTS independently seeded
K-means initializations, each with KMEANS_N_INIT internal starts. Every model receives
the same initial values for a given restart. Restart 0 preserves the previous data-seed
initialization; extra restarts use data_seed + 10000 * restart. These are repeated fits
of the same data, not additional simulated datasets. Exact fixed-parameter likelihoods
are also retained. The defaults now require 576 mixture fits, including 36 dense fits.

In [ ]:
SEED = 0
NPC_PATH = Path('/grp01/sbms_yhhuang/lingyu/STFoundation/Data/NPC/patient1_adata_all_pvalue.h5ad')
N_LOCATIONS = 600
N_GENES = 120
K = 4  # Four manually defined mean profiles below.
N_REPLICATES = 3
LENGTH_SCALES = [0.08, 0.20]
SIGMA_SQS = [0.1, 0.5]
TRUE_DELTA = 0.5
THREADS = 1
RUN_DENSE_FIT = True
N_RESTARTS = 3
KMEANS_N_INIT = 10

FIT_OPTIONS = dict(iter=50, threshold=1e-4, likelihood_tolerance=1e-6,
                   covariance_update='line_search', delta_bounds=(1e-5, 10.0),
                   search_grid_size=12, search_refinements=3,
                   search_xatol=1e-4, scale_floor=1e-8,
                   acceptance_tolerance=1e-10)
INIT_SIGMA_SQ = 0.5
INIT_DELTA = 1.0

random.seed(SEED)
np.random.seed(SEED)

In [ ]:
BASE = dict(partition='axis', order='native', group_size=16, d=5,
            partition_seed=0, order_seed=SEED, reference='default')

CASES = [dict(name='default', **BASE)]
for b in (8, 32, 64):
    CASES.append(dict(BASE, name='block_' + str(b), group_size=b))
for d in (0, 2, 10):
    CASES.append(dict(BASE, name='neighbors_' + str(d), d=d))
CASES.append(dict(BASE, name='joint_kmeans', partition='joint',
                  order='raster', reference='raster'))
for seed in (1, 2):
    CASES.append(dict(BASE, name='partition_seed_' + str(seed), partition_seed=seed,
                      order='raster', reference='raster'))
for order in ('raster', 'maximin'):
    CASES.append(dict(BASE, name=order, order=order))
for seed in (0, 1, 2):
    CASES.append(dict(BASE, name='random_order_' + str(seed),
                      order='random', order_seed=seed))

cases = ([dict(BASE, name='dense_fit', partition='dense')] if RUN_DENSE_FIT else []) + CASES
print('Simulated datasets:', N_REPLICATES * len(LENGTH_SCALES) * len(SIGMA_SQS))
print('Mixture fits:', N_REPLICATES * len(LENGTH_SCALES) * len(SIGMA_SQS) * len(cases) * N_RESTARTS)
pd.DataFrame(cases)

In [ ]:
bench = BenchmarkRecorder('npc_sensitivity.ipynb', seed=SEED)
results_dir = bench.output_dir / ('npc_sensitivity_' + bench.session_id)
results_dir.mkdir(exist_ok=True)
config = dict(seed=SEED, npc_path=str(NPC_PATH), n_locations=N_LOCATIONS,
              n_genes=N_GENES, K=K,
              replicates=N_REPLICATES, restarts=N_RESTARTS, kmeans_n_init=KMEANS_N_INIT,
              length_scales=LENGTH_SCALES,
              sigma_sqs=SIGMA_SQS, true_delta=TRUE_DELTA,
              mean_profiles='x-gradient, y-gradient, localized peak, diagonal band; not standardized',
              threads=THREADS, fit_options=FIT_OPTIONS,
              init_sigma_sq=INIT_SIGMA_SQ, init_delta=INIT_DELTA, cases=cases,
              coordinate_normalization='center, then divide both axes by maximum extent',
              kernel='rbf', gamma_definition='1 / (2 * length_scale**2)',
              covariance_definition='sigma_sq * (C + delta * I)',
              data_seed_definition='SEED + replicate',
              init_seed_definition='data_seed + 10000 * restart',
              restart_selection='highest final likelihood within each dataset and model',
              multistart_timing='one setup + all fits + all shared K-means initializations',
              memory_scope='sampled notebook process-tree RSS, not isolated method memory')
(results_dir / 'configuration.json').write_text(json.dumps(config, indent=2) + '\n')
print('Environment:', bench.environment_path)
print('Run records:', bench.runs_path)
print('Tables, fitted arrays and figures:', results_dir)

## NPC coordinates

Only obs['x'] and obs['y'] are read; the measured feature values are not used.
A fixed seeded subset is reused throughout. Start with N_LOCATIONS = 600; increasing it
also increases the memory needed for full-covariance simulation and the dense reference.

In [ ]:
adata_impt_all = sc.read_h5ad(NPC_PATH, backed='r')
all_spatial = adata_impt_all.obs[['x', 'y']].to_numpy(dtype=float)
all_ids = adata_impt_all.obs.index.to_numpy(dtype=str)
adata_impt_all.file.close()

n = len(all_spatial) if N_LOCATIONS is None else min(N_LOCATIONS, len(all_spatial))
location_index = np.sort(np.random.default_rng(SEED).choice(len(all_spatial), n, replace=False))
raw_spatial = all_spatial[location_index]
center = raw_spatial.mean(axis=0)
extent = float(np.ptp(raw_spatial, axis=0).max())
spatialxy = (raw_spatial - center) / extent

pd.DataFrame({'source_index': location_index, 'obs_id': all_ids[location_index],
              'raw_x': raw_spatial[:, 0], 'raw_y': raw_spatial[:, 1],
              'x': spatialxy[:, 0], 'y': spatialxy[:, 1]}).to_csv(
                  results_dir / 'coordinates.csv', index=False)
np.savez_compressed(results_dir / 'coordinates.npz', spatial=spatialxy,
                    source_index=location_index, center=center, extent=extent)
print('Selected / available locations:', len(spatialxy), '/', len(all_spatial))
print('One dense float64 matrix (MiB):', n * n * 8 / 2**20)
del all_spatial, all_ids, adata_impt_all


## Simulation and partition helpers

Four mean profiles are specified explicitly in make_mean_profiles: an x-gradient,
a y-gradient, a localized peak and a diagonal band. Their offsets and amplitudes are
not standardized. The means are fixed across seeds, length scales and covariance scales;
replicates vary the gene assignments and Gaussian residuals, not the true patterns.
The formulas use the normalized coordinates and are editable directly below.
Genes have balanced known component labels and independent full-covariance Gaussian
residuals with covariance sigma_sq * (C + delta * I). SIGMA_SQS directly specifies the
covariance-scale values; no residual-SD conversion is used.
These profiles are continuous and may be negative; no count transformation is applied.

Axis partitioning reproduces the current code, including fixed k-means settings.
Joint 2-D k-means uses the same number of nonempty blocks as the corresponding axis
partition, but block sizes and shapes can differ. Partition-method and partition-seed
comparisons use raster ordering on both sides, with the axis/raster case as their matched
reference, so arbitrary k-means label order is not an extra varying choice.
Raster, random and maximin comparisons
reuse identical block membership. Each order rebuilds nearest-predecessor dependencies.
Maximin begins at the centroid nearest the tissue center. Native order is k-means-label
enumeration, not a geometrically sorted raster.

In [ ]:
def make_mean_profiles(spatial):
    x, y = spatial.T
    return np.column_stack([
        1.0 + 2.0 * x,
        1.0 + 2.0 * y,
        0.5 + 2.0 * np.exp(-((x - 0.15)**2 + (y + 0.10)**2) / (2 * 0.12**2)),
        0.5 + 1.5 * np.exp(-(x + y)**2 / (2 * 0.10**2)),
    ])


def simulate_profiles(spatial, length_scale, sigma_value, seed):
    rng = np.random.default_rng(seed)
    covariance = rbf_kernel(spatial, gamma=1 / (2 * length_scale**2))
    means = make_mean_profiles(spatial)
    labels = np.arange(N_GENES) % K
    rng.shuffle(labels)
    sigma_sq = np.full(K, sigma_value)
    delta = np.full(K, TRUE_DELTA)
    residual_factor = np.linalg.cholesky(covariance + TRUE_DELTA * np.eye(len(spatial)))
    Y = means[:, labels] + np.sqrt(sigma_sq[0]) * (
        residual_factor @ rng.standard_normal((len(spatial), N_GENES)))
    return Y, labels, means, covariance, sigma_sq, delta


def axis_partition(spatial, group_size, seed):
    group = len(spatial) / group_size
    g_r = max(1, int(np.sqrt(group)))
    g_c = max(1, int(group / g_r))
    rows = KMeans(n_clusters=g_r, random_state=seed, n_init=10).fit_predict(spatial[:, :1])
    cols = KMeans(n_clusters=g_c, random_state=seed, n_init=10).fit_predict(spatial[:, 1:])
    return [np.flatnonzero((rows == r) & (cols == c))
            for r in range(g_r) for c in range(g_c) if np.any((rows == r) & (cols == c))]


def make_blocks(spatial, case):
    if case['partition'] == 'dense':
        return [np.arange(len(spatial))], [[]]
    blocks = axis_partition(spatial, case['group_size'], case['partition_seed'])
    if case['partition'] == 'joint':
        labels = KMeans(n_clusters=len(blocks), random_state=case['partition_seed'],
                        n_init=10).fit_predict(spatial)
        blocks = [np.flatnonzero(labels == i) for i in range(len(blocks))]
    elif case['partition'] != 'axis':
        raise ValueError('Unknown partition')
    centers = np.array([spatial[block].mean(axis=0) for block in blocks])
    m = len(blocks)
    if case['order'] == 'native':
        order = np.arange(m)
    elif case['order'] == 'raster':
        order = np.lexsort((centers[:, 1], centers[:, 0]))
    elif case['order'] == 'random':
        order = np.random.default_rng(case['order_seed']).permutation(m)
    elif case['order'] == 'maximin':
        distances = distance_matrix(centers, centers)
        first = int(np.argmin(np.sum((centers - centers.mean(axis=0))**2, axis=1)))
        order = [first]
        minimum = distances[first].copy()
        minimum[first] = -np.inf
        while len(order) < m:
            selected = int(np.argmax(minimum))
            order.append(selected)
            minimum = np.minimum(minimum, distances[selected])
            minimum[order] = -np.inf
    else:
        raise ValueError('Unknown order')
    blocks = [blocks[i] for i in order]
    centers = np.array([spatial[block].mean(axis=0) for block in blocks])
    distances = distance_matrix(centers, centers)
    dependencies = [list(range(i)) if i <= case['d']
                    else distances[i, :i].argsort()[:case['d']].tolist()
                    for i in range(m)]
    return blocks, dependencies


def block_summary(blocks, dependencies, spatial):
    sizes = np.array([len(block) for block in blocks])
    conditioning = [sum(sizes[j] for j in deps) for deps in dependencies]
    diameters = [float(distance_matrix(spatial[b], spatial[b]).max()) for b in blocks]
    return dict(n_blocks=len(blocks), block_size_min=int(sizes.min()),
                block_size_mean=float(sizes.mean()), block_size_max=int(sizes.max()),
                block_size_cv=float(sizes.std() / sizes.mean()),
                conditioning_size_mean=float(np.mean(conditioning)),
                conditioning_size_max=int(max(conditioning)),
                block_diameter_mean=float(np.mean(diameters)))


In [ ]:
def exact_log_densities(Y, covariance, mean, sigma_sq, delta):
    n, g = Y.shape
    ll = np.empty((g, mean.shape[1]))
    for k in range(mean.shape[1]):
        factor = np.linalg.cholesky(covariance + delta[k] * np.eye(n))
        residual = solve_triangular(factor, Y - mean[:, k, None],
                                    lower=True, check_finite=False)
        ll[:, k] = -0.5 * (n * np.log(2 * np.pi * sigma_sq[k])
                           + 2 * np.log(np.diag(factor)).sum()
                           + np.square(residual).sum(axis=0) / sigma_sq[k])
    return ll


def likelihood_metrics(exact, approximate, labels, n):
    true_error = (approximate - exact)[np.arange(len(labels)), labels]
    mixture_error = logsumexp(approximate, axis=1) - logsumexp(exact, axis=1)
    return dict(true_component_ll_rmse_per_location=float(np.sqrt(np.mean(true_error**2)) / n),
                mixture_ll_rmse_per_location=float(np.sqrt(np.mean(mixture_error**2)) / n),
                mixture_ll_bias_per_location=float(np.mean(mixture_error) / n),
                fixed_parameter_label_agreement=float(
                    np.mean(approximate.argmax(axis=1) == exact.argmax(axis=1))))


def pattern_metrics(truth, fitted):
    a = truth - truth.mean(axis=0)
    b = fitted - fitted.mean(axis=0)
    denominator = np.linalg.norm(a, axis=0)[:, None] * np.linalg.norm(b, axis=0)[None, :]
    correlations = np.divide(a.T @ b, denominator, out=np.zeros_like(denominator),
                             where=denominator > 0)
    rows, cols = linear_sum_assignment(-correlations)
    return dict(pattern_correlation=float(correlations[rows, cols].mean()),
                pattern_rmse=float(np.sqrt(np.mean((truth[:, rows] - fitted[:, cols])**2))))


def latest_resources():
    record = json.loads(bench.runs_path.read_text().splitlines()[-1])
    return dict(peak_rss_mib=record['peak_process_tree_rss_bytes'] / 2**20,
                baseline_rss_mib=record['baseline_process_tree_rss_bytes'] / 2**20)


def fit_diagnostics(model):
    history = np.asarray(model.log_likelihood_history)
    steps = [step for iteration in model.line_search_diagnostics for step in iteration]
    return dict(iterations=len(history) - 1,
                iteration_limit_reached=len(history) - 1 >= FIT_OPTIONS['iter'],
                final_log_likelihood=float(history[-1]),
                likelihood_decreases=int(np.sum(np.diff(history) < -1e-8 * np.maximum(1, np.abs(history[:-1])))),
                search_evaluations=sum(step.get('evaluations', 0) for step in steps),
                rejected_updates=sum(not step['accepted'] for step in steps),
                boundary_updates=sum(step.get('at_boundary', False) for step in steps))


def select_restarts(results):
    # Likelihoods select starts within a model, never across approximations.
    grouped = results.groupby(['dataset', 'name'], sort=False)
    best = results.loc[grouped.final_log_likelihood.idxmax()].copy()
    budget = grouped.agg(restarts_completed=('restart', 'count'),
                         all_fit_seconds=('fit_seconds', 'sum'),
                         all_init_seconds=('init_seconds', 'sum'),
                         multistart_fit_peak_rss_mib=('fit_peak_rss_mib', 'max'),
                         restart_ari_min=('ari', 'min'), restart_ari_max=('ari', 'max'),
                         restart_ari_std=('ari', 'std'))
    best = best.merge(budget, on=['dataset', 'name'], validate='one_to_one')
    best['multistart_seconds'] = (best.setup_seconds + best.all_fit_seconds
                                 + best.all_init_seconds)
    return best

## Preview

This preview uses the first planned simulation. The actual experiment regenerates it with
the same seed. Colors in partition maps identify block order; connecting lines join
successive block centroids, not conditional-neighbor edges.

In [ ]:
with threadpool_limits(limits=THREADS):
    preview_Y, preview_labels, preview_mean, _, _, _ = simulate_profiles(
        spatialxy, LENGTH_SCALES[0], SIGMA_SQS[0], SEED)
    fig, axes = plt.subplots(1, K, figsize=(3 * K, 3), dpi=150, squeeze=False)
    for k, ax in enumerate(axes[0]):
        ax.scatter(spatialxy[:, 0], spatialxy[:, 1], c=preview_mean[:, k],
                   s=8, cmap='viridis', vmin=preview_mean.min(), vmax=preview_mean.max())
        ax.set(title='True pattern ' + str(k), aspect='equal')
    fig.tight_layout()
    fig.savefig(results_dir / 'true_patterns_preview.png')
    plt.show()
    plt.close(fig)

    fig, axes = plt.subplots(1, 4, figsize=(12, 3), dpi=150)
    for ax, order in zip(axes, ('native', 'raster', 'random', 'maximin')):
        blocks, deps = make_blocks(spatialxy, dict(BASE, order=order))
        block_id = np.empty(len(spatialxy), dtype=int)
        centers = []
        for i, block in enumerate(blocks):
            block_id[block] = i
            centers.append(spatialxy[block].mean(axis=0))
        centers = np.asarray(centers)
        ax.scatter(*spatialxy.T, c=block_id, s=8, cmap='viridis')
        ax.plot(*centers.T, color='black', linewidth=0.5, alpha=0.5)
        ax.set(title=order, aspect='equal')
    fig.tight_layout()
    fig.savefig(results_dir / 'partition_order_preview.png')
    plt.show()
    plt.close(fig)
del preview_Y, preview_labels, preview_mean, blocks, deps, _


## Run sensitivity experiments

For each dataset, the exact likelihood uses the known generating mean/covariance and
uniform mixture weights. The approximation is evaluated at those same parameters.
For each restart, all models receive identical K-means means, uniform weights, scale
and nugget. The dense fit uses one block containing every location, with no conditional
approximation. Length scale is held at its true value: this tests partition/order
sensitivity, not length-scale estimation. Means, weights, scale and nugget are fitted
with the same line-search settings. K-means labels and accuracy are saved before fitting.

Setup includes partition/dependency construction, Kernel initialization and the
fixed-parameter likelihood. Each case reuses its Kernel across restarts; all fitted
parameters and histories are reset. Fit timing covers run_cluster only. K-means has
its own timer. The multistart budget counts one setup, all fits and all initializations
for each model, although the shared initializations are physically computed only once.
Generation, the separate dense reference, saving and post-fit evaluation are excluded.
RAM is sampled whole-notebook RSS, including shared simulation/reference arrays; it is
not an isolated-method memory benchmark. Dense covariance is retained for reference,
but is not passed to the sparse model constructor.

Results are saved after each completed fit. An error stops the cell with its normal
traceback; benchmarked errors also remain in the resource log. A fit hitting the
iteration limit is not labeled converged.
Rerunning this cell replaces these session tables; rerun setup for a separate session.

In [ ]:
rows = []
initialization_rows = []
with threadpool_limits(limits=THREADS):
    for replicate in range(N_REPLICATES):
        data_seed = SEED + replicate
        for length_scale in LENGTH_SCALES:
            for sigma_value in SIGMA_SQS:
                dataset = 'rep{}_ell{:g}_sigma_sq{:g}'.format(replicate, length_scale, sigma_value)
                dataset_dir = results_dir / dataset
                dataset_dir.mkdir(exist_ok=True)
                common = dict(dataset=dataset, replicate=replicate, data_seed=data_seed,
                              length_scale=length_scale, sigma_sq=sigma_value,
                              N=len(spatialxy), G=N_GENES, K=K, threads=THREADS)
                print(dataset)
                with bench.run('Full GP simulation', dataset,
                               'Generate full-covariance profiles', parameters=common, device='cpu'):
                    Y, truth, true_mean, true_cov, true_sigma, true_delta = simulate_profiles(
                        spatialxy, length_scale, sigma_value, data_seed)
                np.savez_compressed(dataset_dir / 'simulation.npz', Y=Y, labels=truth,
                                    true_mean=true_mean, true_sigma_sq=true_sigma, true_delta=true_delta)
                initializations = []
                for restart in range(N_RESTARTS):
                    init_seed = data_seed + 10000 * restart
                    restart_dir = dataset_dir / ('restart_' + str(restart))
                    restart_dir.mkdir(exist_ok=True)
                    init_row = dict(common, restart=restart, init_seed=init_seed,
                                    kmeans_n_init=KMEANS_N_INIT)
                    with bench.run('Shared K-means initialization', dataset, 'K-means fit only',
                                   parameters=init_row, input_shape=Y.shape, device='cpu') as run:
                        start = time.perf_counter()
                        kmeans = KMeans(n_clusters=K, random_state=init_seed,
                                        n_init=KMEANS_N_INIT).fit(Y.T)
                        init_row['init_seconds'] = time.perf_counter() - start
                        run.finish_timing(init_row['init_seconds'])
                    initial_mean = kmeans.cluster_centers_.T.copy()
                    initial_labels = kmeans.labels_.copy()
                    init_row.update(initial_ari=adjusted_rand_score(truth, initial_labels),
                                    initial_nmi=normalized_mutual_info_score(truth, initial_labels),
                                    initial_inertia=float(kmeans.inertia_))
                    init_row.update({'initial_' + key: value for key, value in
                                     pattern_metrics(true_mean, initial_mean).items()})
                    np.savez_compressed(restart_dir / 'initialization.npz', mean=initial_mean,
                                        labels=initial_labels, init_seed=init_seed,
                                        pi=np.ones(K) / K, sigma_sq=np.full(K, INIT_SIGMA_SQ),
                                        delta=np.full(K, INIT_DELTA))
                    initializations.append((init_row, initial_mean, initial_labels))
                    initialization_rows.append(init_row)
                    pd.DataFrame(initialization_rows).to_csv(results_dir / 'initialization_runs.csv', index=False)
                del kmeans
                with bench.run('Dense fixed-parameter likelihood', dataset,
                               'Dense Cholesky and log densities; excludes generation of C',
                               parameters=common, input_shape=Y.shape, device='cpu'):
                    exact_ll = exact_log_densities(Y, true_cov, true_mean, true_sigma, true_delta)
                np.savez_compressed(dataset_dir / 'exact_likelihood.npz', log_densities=exact_ll)
                predictions = {}
                dataset_rows = []
                for case in cases:
                    setup_row = dict(common, **case)
                    random.seed(data_seed)
                    np.random.seed(data_seed)
                    with bench.run('SparseAEH setup and likelihood', dataset,
                                   'Partition, dependencies, Kernel and fixed-parameter likelihood',
                                   parameters=dict(common, **case), input_shape=Y.shape,
                                   device='cpu') as run:
                        start = time.perf_counter()
                        blocks, dependencies = make_blocks(spatialxy, case)
                        model = MixedGaussian(spatialxy, ss_loc=blocks, dependency=dependencies,
                                              kernel='rbf', l=1 / (2 * length_scale**2))
                        approximate_ll = GaussianNLL(Y, model.kernel, true_mean, true_sigma, true_delta)
                        setup_row['setup_seconds'] = time.perf_counter() - start
                        run.finish_timing(setup_row['setup_seconds'])
                    setup_row.update({'setup_' + key: value for key, value in latest_resources().items()})
                    setup_row.update(block_summary(blocks, dependencies, spatialxy))
                    setup_row.update(likelihood_metrics(exact_ll, approximate_ll, truth, len(spatialxy)))
                    partition = dict(blocks=[b.tolist() for b in blocks], dependencies=dependencies)
                    (dataset_dir / (case['name'] + '_partition.json')).write_text(json.dumps(partition) + '\n')
                    for init_row, initial_mean, initial_labels in initializations:
                        restart, init_seed = init_row['restart'], init_row['init_seed']
                        print(' ', case['name'], 'restart', restart, 'initial ARI', round(init_row['initial_ari'], 4))
                        restart_dir = dataset_dir / ('restart_' + str(restart))
                        row = dict(setup_row, **{key: value for key, value in init_row.items() if key not in common})
                        random.seed(init_seed)
                        np.random.seed(init_seed)
                        with bench.run('SparseAEH mixture fit', dataset,
                                       'run_cluster only, reusing timed Kernel setup',
                                       parameters=dict(common, **case, restart=restart, init_seed=init_seed,
                                                       fit_options=FIT_OPTIONS,
                                                       init_sigma_sq=INIT_SIGMA_SQ, init_delta=INIT_DELTA),
                                       input_shape=Y.shape, device='cpu') as run:
                            start = time.perf_counter()
                            model.run_cluster(Y, K, mean=initial_mean.copy(), pi=np.ones(K) / K,
                                              sigma_sq=np.full(K, INIT_SIGMA_SQ),
                                              delta=np.full(K, INIT_DELTA), **FIT_OPTIONS)
                            row['fit_seconds'] = time.perf_counter() - start
                            run.finish_timing(row['fit_seconds'])
                            run.details.update(fit_diagnostics(model))
                        row.update({'fit_' + key: value for key, value in latest_resources().items()})
                        if not (np.isfinite(model.mean).all() and np.isfinite(model.omega).all()
                                and np.isfinite(model.log_likelihood_history).all()):
                            raise ValueError('Non-finite fitted outputs')
                        row.update(ari=adjusted_rand_score(truth, model.labels),
                                   nmi=normalized_mutual_info_score(truth, model.labels),
                                   total_seconds=row['setup_seconds'] + row['fit_seconds'])
                        row['ari_gain_from_init'] = row['ari'] - row['initial_ari']
                        row['ari_vs_initial'] = adjusted_rand_score(initial_labels, model.labels)
                        row.update(pattern_metrics(true_mean, model.mean))
                        row.update(fit_diagnostics(model))
                        np.savez_compressed(restart_dir / (case['name'] + '_fit.npz'),
                                            labels=model.labels, mean=model.mean, omega=model.omega,
                                            sigma_sq=model.sigma_sq, delta=model.delta, pi=model.pi,
                                            likelihood_history=model.log_likelihood_history,
                                            fixed_log_densities=approximate_ll, restart=restart, init_seed=init_seed)
                        (restart_dir / (case['name'] + '_search.json')).write_text(
                            json.dumps(model.line_search_diagnostics, indent=2) + '\n')
                        predictions[(restart, case['name'])] = model.labels.copy()
                        rows.append(row)
                        dataset_rows.append(row)
                        pd.DataFrame(rows).to_csv(results_dir / 'sensitivity_runs.csv', index=False)
                    del model
                    gc.collect()
                for row in dataset_rows:
                    restart = row['restart']
                    for reference in ('default', 'dense_fit', row['reference']):
                        if (restart, reference) in predictions:
                            agreement = adjusted_rand_score(predictions[(restart, reference)],
                                                            predictions[(restart, row['name'])])
                            row['ari_vs_' + reference] = agreement
                            if reference == row['reference']:
                                row['ari_vs_matched_reference'] = agreement
                pd.DataFrame(rows).to_csv(results_dir / 'sensitivity_runs.csv', index=False)
                del Y, true_cov, true_mean, true_sigma, true_delta, exact_ll, initializations, initial_mean
                gc.collect()

results = pd.DataFrame(rows)
print('Saved:', results_dir / 'sensitivity_runs.csv')
results

## Tables and plots

initialization_runs.csv reports K-means ARI, NMI, pattern scores and inertia for each
dataset/restart. sensitivity_runs.csv keeps every fit, its initial scores, ARI gain and
agreement with the dense/default fit from that same start. Paired-change tables match
both dataset and restart; partition-method/seed cases retain the raster reference.

best_restart_runs.csv selects the highest final likelihood separately for each dataset
and model. Truth labels are used only for evaluation, never selection. Likelihood values
from different approximations are not used to choose between models. The selected row's
ari_vs_dense_fit still compares against the dense fit from the same restart, not the
independently selected dense winner. Restart ranges/SDs remain available in this table.

The main summary and plots use one selected fit per dataset/model. Mean and SD are across
simulated datasets, not across restarts, and are not confidence intervals. Initialization
summaries first average starts within each dataset. The timing panel includes one setup,
all fits and all shared initializations; it does not show only the winning fit's cost.
Raw total_seconds remains setup + one fit, excluding initialization, and repeated setup
values must not be summed across restarts. Peak RSS in the summary is the maximum across
all fits, not just the selected one. It remains whole-notebook, not isolated-method memory.

Accuracy is measured across genes, not locations. Pattern scores use one-to-one
correlation matching to remove component-label permutations. Stability is not accuracy;
inspect initial and fitted scores, restart variability and iteration-limited runs together.

In [ ]:
results = pd.read_csv(results_dir / 'sensitivity_runs.csv')
initialization_results = pd.read_csv(results_dir / 'initialization_runs.csv')
initial_metrics = ['initial_ari', 'initial_nmi', 'initial_pattern_correlation', 'initial_pattern_rmse']
initial_by_dataset = initialization_results.groupby(
    ['length_scale', 'sigma_sq', 'dataset'])[initial_metrics].mean().reset_index()
initialization_summary = initial_by_dataset.groupby(['length_scale', 'sigma_sq'])[initial_metrics].agg(['mean', 'std', 'count'])
initialization_summary.columns = ['_'.join(column) for column in initialization_summary.columns]
initialization_summary.to_csv(results_dir / 'initialization_summary.csv')

best_results = select_restarts(results)
best_results.to_csv(results_dir / 'best_restart_runs.csv', index=False)
metrics = ['initial_ari', 'ari', 'ari_gain_from_init', 'nmi', 'pattern_correlation', 'pattern_rmse',
           'restart_ari_min', 'restart_ari_max', 'restart_ari_std',
           'true_component_ll_rmse_per_location', 'mixture_ll_rmse_per_location',
           'fixed_parameter_label_agreement', 'multistart_seconds', 'multistart_fit_peak_rss_mib',
           'block_size_mean', 'block_size_max', 'conditioning_size_mean',
           'iterations', 'iteration_limit_reached']
summary = best_results.groupby(['length_scale', 'sigma_sq', 'name'])[metrics].agg(['mean', 'std', 'count'])
summary.columns = ['_'.join(column) for column in summary.columns]
summary.to_csv(results_dir / 'sensitivity_summary.csv')

paired_metrics = ['ari', 'pattern_correlation', 'mixture_ll_rmse_per_location', 'total_seconds']
pair_keys = ['dataset', 'restart']
for reference_name, filename, suffix in [('default', 'paired_changes.csv', 'default'),
                                          ('dense_fit', 'dense_paired_changes.csv', 'dense')]:
    reference = results[results.name == reference_name][pair_keys + paired_metrics]
    if reference.empty:
        continue
    paired = results.merge(reference, on=pair_keys, suffixes=('', '_' + suffix), validate='many_to_one')
    for metric in paired_metrics:
        paired[metric + '_change_from_' + suffix] = paired[metric] - paired[metric + '_' + suffix]
    paired.to_csv(results_dir / filename, index=False)
matched_reference = results[pair_keys + ['name'] + paired_metrics].rename(columns={'name': 'reference'})
matched = results.merge(matched_reference, on=pair_keys + ['reference'],
                       suffixes=('', '_reference'), validate='many_to_one')
for metric in paired_metrics:
    matched[metric + '_change_from_reference'] = matched[metric] - matched[metric + '_reference']
matched.to_csv(results_dir / 'matched_reference_changes.csv', index=False)
print('Fits reaching iteration limit:', int(results.iteration_limit_reached.sum()))
print('Initial K-means performance (starts averaged within each dataset):')
print(initialization_summary.to_string())
summary

In [ ]:
plot_metrics = [('ari', 'Gene clustering ARI'),
                ('pattern_correlation', 'Matched pattern correlation'),
                ('mixture_ll_rmse_per_location', 'Fixed-parameter log-likelihood RMSE / location'),
                ('multistart_seconds', 'One setup + all fits + all initializations (s)')]
order = [case['name'] for case in cases]
for length_scale in LENGTH_SCALES:
    for sigma_value in SIGMA_SQS:
        subset = best_results[(best_results.length_scale == length_scale)
                                 & (best_results.sigma_sq == sigma_value)]
        if subset.empty:
            continue
        fig, axes = plt.subplots(2, 2, figsize=(14, 9), dpi=150)
        for ax, (metric, label) in zip(axes.flat, plot_metrics):
            stats = subset.groupby('name')[metric].agg(['mean', 'std']).reindex(order)
            present = stats['mean'].notna()
            x = np.arange(len(order))[present]
            ax.errorbar(x, stats.loc[present, 'mean'],
                        yerr=stats.loc[present, 'std'].fillna(0),
                        fmt='o', capsize=3, color='teal')
            if metric == 'ari':
                initial_stats = subset.groupby('name').initial_ari.agg(['mean', 'std']).reindex(order)
                ax.errorbar(x, initial_stats.loc[present, 'mean'],
                            yerr=initial_stats.loc[present, 'std'].fillna(0),
                            fmt='x', capsize=3, color='gray', label='Initial K-means (selected start)')
                ax.legend(fontsize=8)
            ax.set_xticks(np.arange(len(order)), order, rotation=55, ha='right', fontsize=8)
            ax.set_ylabel(label)
            ax.grid(axis='y', alpha=0.2)
        fig.suptitle('Length scale {:g}; sigma_sq {:g}; likelihood-selected starts; mean +/- SD across datasets'.format(length_scale, sigma_value))
        fig.tight_layout()
        for suffix in ('png', 'pdf'):
            fig.savefig(results_dir / ('sensitivity_ell{:g}_sigma_sq{:g}.'.format(length_scale, sigma_value) + suffix))
        plt.show()
        plt.close(fig)

## Scope still open

This notebook does not yet assess count-model misspecification, real biological labels,
length-scale estimation, rare gene groups, variable density/resolution sweeps, or
fixed-physical-radius conditioning. It does not establish isolated-process memory scaling.
The stored partitions, fit parameters, search diagnostics, resource logs and seeds
support auditing the experiments actually run. Manuscript/reviewer TODOs remain open
until the remote results are inspected. If ARI is uniformly near one, that can indicate an
easy simulation rather than demonstrated robustness; inspect likelihood/pattern errors
and consider higher noise or lower pattern contrast before choosing the final design.